# Week 10 — Thursday Lab Notebook
## Applied DL Mini-Track (Image **and** Text)

**Topic focus:** applied deep learning workflow for image data and text data, practical training, evaluation, and choosing the right approach

This notebook is written in simple language for lab teaching.  
We will use **PyTorch** for both tracks so students can see one consistent workflow.

## 3-Hour Structure

**Hour 1**
- understand the image mini-track
- prepare image data
- build and train a small CNN
- evaluate image predictions

**Hour 2**
- understand the text mini-track
- convert text into integer tokens
- build and train a small embedding-based classifier
- evaluate text predictions

**Hour 3**
- compare image workflow and text workflow
- discuss practical evaluation
- discuss common mistakes
- solve small prediction tasks

## Part A — What Does “Applied DL” Mean?

So far, we studied neural networks, model building, and improving training.

Today we move one step ahead.

We will apply deep learning to **two real data types**:

1. **Images**  
2. **Text**

This is important because real projects do not all look the same.

- image data has **height, width, pixels**
- text data has **words, sequence, vocabulary**

The model structure changes because the data structure changes.

But the overall workflow is still similar:

- prepare data
- split data
- build model
- train model
- validate model
- test model
- interpret results

## Part B — Import Libraries

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Part C — General Training Helpers

We will use one training function for both mini-tracks.

That is useful because students can see that the **workflow stays the same** even when the data changes.

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10, device=device):
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss_total = 0
        train_correct = 0
        train_total = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

            train_loss_total += loss.item() * xb.size(0)
            preds = outputs.argmax(dim=1)
            train_correct += (preds == yb).sum().item()
            train_total += yb.size(0)

        train_loss = train_loss_total / train_total
        train_acc = train_correct / train_total

        model.eval()
        val_loss_total = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                outputs = model(xb)
                loss = criterion(outputs, yb)

                val_loss_total += loss.item() * xb.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == yb).sum().item()
                val_total += yb.size(0)

        val_loss = val_loss_total / val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

    return history


def plot_history(history, title_prefix="Model"):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], marker="o", label="Train Loss")
    plt.plot(epochs, history["val_loss"], marker="o", label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} - Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], marker="o", label="Train Accuracy")
    plt.plot(epochs, history["val_acc"], marker="o", label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title_prefix} - Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()


def evaluate_model(model, test_loader, class_names=None, title="Confusion Matrix", device=device):
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            outputs = model(xb)
            preds = outputs.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds.tolist())
            all_true.extend(yb.numpy().tolist())

    acc = accuracy_score(all_true, all_preds)
    cm = confusion_matrix(all_true, all_preds)

    print("Test Accuracy:", round(acc, 4))
    print("
Classification Report:
")
    if class_names is not None:
        print(classification_report(all_true, all_preds, target_names=class_names))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    else:
        print(classification_report(all_true, all_preds))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    plt.title(title)
    plt.show()

    return acc, cm, all_true, all_preds

# Mini-Track 1 — Image Classification

## Part D — Load an Image Dataset

We will use the **Digits** dataset from scikit-learn.

Why this dataset?

- it is already available locally
- it is small and fast
- each image is only **8 × 8** pixels
- there are 10 classes: digits **0 to 9**

This is good for classroom learning.

In [ ]:
digits = load_digits()
X_images = digits.images
y_images = digits.target
image_class_names = [str(i) for i in digits.target_names]

print("Image data shape:", X_images.shape)
print("Target shape:", y_images.shape)
print("Classes:", image_class_names)

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(X_images[i], cmap="gray")
    plt.title(f"Label: {y_images[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

### Teaching point

For image data, every sample has **height** and **width**.

A neural network must receive the data in the correct shape.

For a small CNN, we usually use:

`(batch_size, channels, height, width)`

For grayscale images, `channels = 1`.

## Part E — Split and Prepare the Image Data

We will do three splits:

- training set
- validation set
- test set

We also normalize the pixels.

The digits dataset uses values from **0 to 16**.  
So we divide by **16** to bring values closer to **0 to 1**.

In [ ]:
X_train_img, X_temp_img, y_train_img, y_temp_img = train_test_split(
    X_images, y_images, test_size=0.30, random_state=42, stratify=y_images
)

X_val_img, X_test_img, y_val_img, y_test_img = train_test_split(
    X_temp_img, y_temp_img, test_size=0.50, random_state=42, stratify=y_temp_img
)

X_train_img = X_train_img / 16.0
X_val_img = X_val_img / 16.0
X_test_img = X_test_img / 16.0

print("Train shape:", X_train_img.shape)
print("Validation shape:", X_val_img.shape)
print("Test shape:", X_test_img.shape)

In [ ]:
X_train_img_tensor = torch.tensor(X_train_img, dtype=torch.float32).unsqueeze(1)
X_val_img_tensor = torch.tensor(X_val_img, dtype=torch.float32).unsqueeze(1)
X_test_img_tensor = torch.tensor(X_test_img, dtype=torch.float32).unsqueeze(1)

y_train_img_tensor = torch.tensor(y_train_img, dtype=torch.long)
y_val_img_tensor = torch.tensor(y_val_img, dtype=torch.long)
y_test_img_tensor = torch.tensor(y_test_img, dtype=torch.long)

print("One image tensor shape:", X_train_img_tensor[0].shape)

In [ ]:
image_train_loader = DataLoader(
    TensorDataset(X_train_img_tensor, y_train_img_tensor),
    batch_size=32,
    shuffle=True,
)

image_val_loader = DataLoader(
    TensorDataset(X_val_img_tensor, y_val_img_tensor),
    batch_size=32,
    shuffle=False,
)

image_test_loader = DataLoader(
    TensorDataset(X_test_img_tensor, y_test_img_tensor),
    batch_size=32,
    shuffle=False,
)

xb, yb = next(iter(image_train_loader))
print("Image batch X shape:", xb.shape)
print("Image batch y shape:", yb.shape)

## Part F — CNN Intuition in Easy Words

A **CNN** is useful for image data because nearby pixels matter.

It tries to learn patterns such as:

- edges
- curves
- corners
- simple shapes

A very simple way to explain it:

- early layers learn small visual patterns
- later layers combine them into bigger patterns
- final layer decides the class

Today we will use a **small CNN** so students can see the idea clearly.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

image_model = SimpleCNN().to(device)
print(image_model)

### Why does `16 * 2 * 2` appear?

Our input image is `8 x 8`.

After the first max-pool:
- `8 x 8` becomes `4 x 4`

After the second max-pool:
- `4 x 4` becomes `2 x 2`

At that point, we have **16 feature maps**, each of size **2 x 2**.

So total features = `16 × 2 × 2 = 64`

In [ ]:
image_criterion = nn.CrossEntropyLoss()
image_optimizer = torch.optim.Adam(image_model.parameters(), lr=0.001)

image_history = train_model(
    image_model,
    image_train_loader,
    image_val_loader,
    image_criterion,
    image_optimizer,
    epochs=12,
)

In [ ]:
plot_history(image_history, title_prefix="Image CNN")

In [ ]:
image_test_acc, image_cm, image_true, image_preds = evaluate_model(
    image_model,
    image_test_loader,
    class_names=image_class_names,
    title="Image Track - Digits Confusion Matrix",
)

In [ ]:
# Show some image predictions
image_model.eval()
plt.figure(figsize=(10, 5))

with torch.no_grad():
    sample_x = X_test_img_tensor[:10].to(device)
    sample_outputs = image_model(sample_x)
    sample_preds = sample_outputs.argmax(dim=1).cpu().numpy()

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test_img[i], cmap="gray")
    plt.title(f"T:{y_test_img[i]} P:{sample_preds[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

### Image mini-track summary

In the image track, our workflow was:

- load images
- normalize pixel values
- reshape to `(batch, channel, height, width)`
- build CNN
- train model
- evaluate predictions

This is a good first example of applied deep learning for visual data.

# Mini-Track 2 — Text Classification

## Part G — Create a Small Text Dataset

For text, we need a different kind of preparation.

A computer does not understand raw words directly.

We first convert words into **integer ids**.
Then the model learns **embeddings** for those ids.

Here we will use a small binary text dataset:

- `0 = ham` (normal message)
- `1 = spam` (promotional / suspicious message)

This dataset is intentionally small so students can understand every step.

In [ ]:
text_samples = [
    ("claim your free prize now", 1),
    ("limited time offer click here", 1),
    ("win cash today reply fast", 1),
    ("congratulations you have won a reward", 1),
    ("urgent claim bonus before midnight", 1),
    ("exclusive deal buy one get one", 1),
    ("free voucher waiting for you", 1),
    ("click this link to get reward", 1),
    ("cheap package available today", 1),
    ("you are selected for lucky draw", 1),
    ("special offer only for today", 1),
    ("reply now to receive gift", 1),
    ("claim your free mobile recharge", 1),
    ("win a shopping coupon now", 1),
    ("this is your final reward reminder", 1),
    ("book meeting with project team", 0),
    ("please submit the assignment today", 0),
    ("your class starts at ten am", 0),
    ("can we reschedule the meeting", 0),
    ("the report is ready for review", 0),
    ("please call me when you arrive", 0),
    ("let us discuss the lab task", 0),
    ("the homework deadline is tomorrow", 0),
    ("your order has been delivered", 0),
    ("bring your notebook to class", 0),
    ("we will meet after lunch", 0),
    ("please share the updated file", 0),
    ("the lecture slide is uploaded", 0),
    ("team presentation is on friday", 0),
    ("kindly review the attached document", 0),
]

text_df = pd.DataFrame(text_samples, columns=["text", "label"])
text_df.head(10)

In [ ]:
text_df["label_name"] = text_df["label"].map({0: "ham", 1: "spam"})
text_df

## Part H — Tokenization and Vocabulary

For a simple beginner workflow:

- convert text to lowercase
- split sentence into words
- build a vocabulary
- assign each word an integer id
- convert every sentence into a sequence of ids
- pad short sequences so batch shapes become equal

This is not the most advanced NLP pipeline.  
But it is very good for teaching the core idea.

In [ ]:
def simple_tokenize(text):
    return text.lower().split()

all_tokens = []
for text in text_df["text"]:
    all_tokens.extend(simple_tokenize(text))

unique_tokens = sorted(set(all_tokens))

# Reserve 0 for padding and 1 for unknown words
vocab = {"<PAD>": 0, "<UNK>": 1}
for i, token in enumerate(unique_tokens, start=2):
    vocab[token] = i

print("Vocabulary size:", len(vocab))
print("First 15 items:")
print(list(vocab.items())[:15])

In [ ]:
def encode_text(text, vocab):
    tokens = simple_tokenize(text)
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]

encoded_sequences = [encode_text(text, vocab) for text in text_df["text"]]
labels = text_df["label"].tolist()

for i in range(3):
    print("Text:", text_df["text"][i])
    print("Encoded:", encoded_sequences[i])
    print()

In [ ]:
max_len = max(len(seq) for seq in encoded_sequences)
print("Maximum sequence length:", max_len)


def pad_sequence(seq, max_len, pad_value=0):
    return seq + [pad_value] * (max_len - len(seq))

padded_sequences = [pad_sequence(seq, max_len) for seq in encoded_sequences]

X_text = np.array(padded_sequences)
y_text = np.array(labels)

print("Text tensor-ready shape:", X_text.shape)

### Teaching point

In text classification, the input shape is usually:

`(batch_size, sequence_length)`

This is different from image data.

That is why the model architecture also changes.

## Part I — Split and Prepare the Text Data

In [ ]:
X_train_txt, X_temp_txt, y_train_txt, y_temp_txt = train_test_split(
    X_text, y_text, test_size=0.30, random_state=42, stratify=y_text
)

X_val_txt, X_test_txt, y_val_txt, y_test_txt = train_test_split(
    X_temp_txt, y_temp_txt, test_size=0.50, random_state=42, stratify=y_temp_txt
)

print("Train shape:", X_train_txt.shape)
print("Validation shape:", X_val_txt.shape)
print("Test shape:", X_test_txt.shape)

In [ ]:
X_train_txt_tensor = torch.tensor(X_train_txt, dtype=torch.long)
X_val_txt_tensor = torch.tensor(X_val_txt, dtype=torch.long)
X_test_txt_tensor = torch.tensor(X_test_txt, dtype=torch.long)

y_train_txt_tensor = torch.tensor(y_train_txt, dtype=torch.long)
y_val_txt_tensor = torch.tensor(y_val_txt, dtype=torch.long)
y_test_txt_tensor = torch.tensor(y_test_txt, dtype=torch.long)

text_train_loader = DataLoader(
    TensorDataset(X_train_txt_tensor, y_train_txt_tensor),
    batch_size=4,
    shuffle=True,
)
text_val_loader = DataLoader(
    TensorDataset(X_val_txt_tensor, y_val_txt_tensor),
    batch_size=4,
    shuffle=False,
)
text_test_loader = DataLoader(
    TensorDataset(X_test_txt_tensor, y_test_txt_tensor),
    batch_size=4,
    shuffle=False,
)

xb, yb = next(iter(text_train_loader))
print("Text batch X shape:", xb.shape)
print("Text batch y shape:", yb.shape)

## Part J — Embedding Intuition

An **embedding** means a model learns a dense numeric representation for each word id.

Simple idea:

- word id `15` is just a number
- embedding layer turns it into a vector like `[0.2, -0.6, 1.1, ...]`
- the model learns these values during training

So instead of treating words like plain labels, the model learns a useful numeric representation.

In [ ]:
class SimpleTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)                   # (batch, seq_len, embed_dim)
        mask = (x != 0).unsqueeze(-1).float()         # ignore padding
        summed = (embedded * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)
        mean_pooled = summed / lengths
        out = self.fc1(mean_pooled)
        out = self.relu(out)
        out = self.fc2(out)
        return out

text_model = SimpleTextClassifier(vocab_size=len(vocab)).to(device)
print(text_model)

In [ ]:
text_criterion = nn.CrossEntropyLoss()
text_optimizer = torch.optim.Adam(text_model.parameters(), lr=0.01)

text_history = train_model(
    text_model,
    text_train_loader,
    text_val_loader,
    text_criterion,
    text_optimizer,
    epochs=20,
)

In [ ]:
plot_history(text_history, title_prefix="Text Classifier")

In [ ]:
text_test_acc, text_cm, text_true, text_preds = evaluate_model(
    text_model,
    text_test_loader,
    class_names=["ham", "spam"],
    title="Text Track - Spam vs Ham Confusion Matrix",
)

In [ ]:
# Show how embedding output looks for one batch
sample_batch, _ = next(iter(text_train_loader))
with torch.no_grad():
    embedding_output = text_model.embedding(sample_batch.to(device)).cpu()

print("Input token ids shape:", sample_batch.shape)
print("Embedding output shape:", embedding_output.shape)

In [ ]:
def predict_text(message, model, vocab, max_len, device=device):
    model.eval()
    encoded = encode_text(message, vocab)
    padded = pad_sequence(encoded[:max_len], max_len)
    x = torch.tensor([padded], dtype=torch.long).to(device)

    with torch.no_grad():
        outputs = model(x)
        pred = outputs.argmax(dim=1).item()

    return "spam" if pred == 1 else "ham"

examples = [
    "free reward waiting for you",
    "please bring your assignment tomorrow",
    "claim your bonus now",
    "team meeting is after lunch",
]

for msg in examples:
    print(msg, "->", predict_text(msg, text_model, vocab, max_len))

### Text mini-track summary

In the text track, our workflow was:

- collect text samples
- tokenize words
- build vocabulary
- encode words into integers
- pad sequences
- use embedding layer
- train classifier
- evaluate predictions

This is a clean beginner example of applied deep learning for text data.

## Part K — Image Workflow vs Text Workflow

### What stays the same?

Both tracks use:

- train / validation / test split
- batches through `DataLoader`
- a PyTorch model
- loss function
- optimizer
- training loop
- evaluation on unseen data

### What changes?

**Image track**
- input = pixels
- shape = `(batch, channel, height, width)`
- model = CNN
- model tries to learn visual patterns

**Text track**
- input = word ids
- shape = `(batch, sequence_length)`
- model = embedding-based classifier
- model tries to learn word patterns

In [ ]:
comparison_df = pd.DataFrame({
    "Track": ["Image", "Text"],
    "Dataset": ["Digits", "Small Spam/Ham"],
    "Model": ["Simple CNN", "Embedding + Mean Pooling"],
    "Test Accuracy": [round(image_test_acc, 4), round(text_test_acc, 4)],
})
comparison_df

## Part L — Practical Evaluation Checklist

When teaching students to evaluate a deep learning project, ask these questions:

1. Did we split the data correctly?  
2. Is the validation set separate from the test set?  
3. Is the model learning or guessing?  
4. Are train and validation curves both improving?  
5. Is there overfitting?  
6. Is accuracy enough, or do we also need a confusion matrix?  
7. Are some classes harder than others?  
8. Can the model predict correctly on new examples?  

This checklist is useful for both image and text projects.

## Part M — Common Beginner Mistakes

### In image tasks
- forgetting to normalize pixel values
- using wrong tensor shape
- flattening data when the model expects images
- not checking class balance

### In text tasks
- forgetting to pad sequences
- mixing raw strings with tensors
- building vocabulary incorrectly
- not handling unknown words

### In both tasks
- training on test data by mistake
- reporting only training accuracy
- ignoring validation performance
- using too many epochs without checking overfitting

## Part N — Small In-Class Practice

### Practice 1
Change the image model learning rate and see whether convergence becomes faster or less stable.

### Practice 2
Change the text embedding size from `16` to `32` and compare the validation accuracy.

### Practice 3
Add one more convolution layer in the image model or one more linear layer in the text model.

### Practice 4
Use the custom `predict_text()` function on 5 new sentences written by students.

## Part O — 10 After-Lab Tasks

### Task 1
Load the digits dataset and print the image shape, target shape, and class names.

### Task 2
Display 6 sample digit images with their labels.

### Task 3
Split the image dataset into train, validation, and test sets.

### Task 4
Normalize the image pixels and convert them into PyTorch tensors with channel dimension.

### Task 5
Train a small CNN on the digits dataset and plot training and validation curves.

### Task 6
Evaluate the image model on the test set using accuracy and confusion matrix.

### Task 7
Create a small text dataset of at least 20 sentences with two classes.

### Task 8
Build a vocabulary, encode the text, and pad all sequences to equal length.

### Task 9
Train an embedding-based text classifier and report test accuracy.

### Task 10
Write 5 lines comparing the image workflow and text workflow.

## Part P — Optional Homework Challenge

Do both mini-projects more independently.

### Project A — Image
- use the digits dataset
- train two image models with different hyperparameters
- compare validation and test performance
- show 10 predictions with true and predicted labels

### Project B — Text
- build your own small text dataset with at least 30 messages
- create ham and spam labels
- train an embedding-based classifier
- test it on 10 new custom sentences

### Final writing task
Write one short reflection answering:

- which track was easier for you?  
- which one had more preprocessing steps?  
- why do image and text tasks need different model structures?

## Part Q — Final Revision Notes

Today you learned:

- applied deep learning means using DL on a real data type
- image and text workflows share the same training pipeline idea
- image data needs shape handling for CNNs
- text data needs tokenization, vocabulary, encoding, and padding
- embeddings are useful for text representation
- CNNs are useful for image patterns
- evaluation is not only about training accuracy
- confusion matrix helps us see class-wise behavior
- applied deep learning is easier to understand when broken into clear steps